# SQL Exploration with SQLite

## tl;dr
Execute SQL summaries against the same validated launch table and reconcile counts with Pandas.

## Context & Methods
IBM course project, reproduced from attributed historical snapshots. See `data/source_manifest.json` for URLs and hashes.

### Key Assumptions
Course success labels include controlled ocean landings; descriptive associations are not causal effects.

## Data
Run from a clean kernel. All ordinary analysis uses included source snapshots.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / 'falcon9').exists() else Path.cwd().parent
if not (ROOT / 'falcon9').exists():
    raise RuntimeError('Start Jupyter in the repository root or notebooks directory')
sys.path.insert(0, str(ROOT))
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from falcon9.analysis import load_data, group_summary
plt.rcParams.update({'figure.figsize': (9, 4.5), 'font.size': 11, 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.grid': False})
df = load_data()


## Results
### Inspect and execute SQL

In [2]:
from falcon9.analysis import SQL_QUERIES, sql_analysis
outputs = sql_analysis(df)
for name, result in outputs.items():
    print(name, '\n', SQL_QUERIES[name])
    display(result)

site_summary 
 SELECT LaunchSite, COUNT(*) AS launches,
        SUM(Class) AS successes, ROUND(AVG(Class), 4) AS success_rate
        FROM launches GROUP BY LaunchSite ORDER BY launches DESC
year_summary 
 SELECT Year, COUNT(*) AS launches, SUM(Class) AS successes,
        ROUND(AVG(Class), 4) AS success_rate FROM launches GROUP BY Year ORDER BY Year
orbit_summary 
 SELECT Orbit, COUNT(*) AS launches,
        ROUND(AVG(Class), 4) AS success_rate FROM launches GROUP BY Orbit ORDER BY launches DESC
first_ground_success 
 SELECT MIN(Date) AS first_ground_success
        FROM launches WHERE Outcome = 'True RTLS' 
payload_summary 
 SELECT LaunchSite, COUNT(*) AS launches,
        ROUND(AVG(PayloadMass), 2) AS mean_payload_kg
        FROM launches GROUP BY LaunchSite ORDER BY launches DESC


,LaunchSite,launches,successes,success_rate
0,CCAFS SLC 40,55,33,0.6000
1,KSC LC 39A,22,17,0.7727
2,VAFB SLC 4E,13,10,0.7692


,Year,launches,successes,success_rate
0,2010,1,0,0.0000
1,2012,1,0,0.0000
2,2013,3,0,0.0000
3,2014,6,2,0.3333
4,2015,6,2,0.3333
5,2016,8,5,0.6250
6,2017,18,15,0.8333
7,2018,18,11,0.6111
8,2019,10,9,0.9000
9,2020,19,16,0.8421


,Orbit,launches,success_rate
0,GTO,27,0.5185
1,ISS,21,0.6190
2,VLEO,14,0.8571
3,PO,9,0.6667
4,LEO,7,0.7143
5,SSO,5,1.0000
6,MEO,3,0.6667
7,SO,1,0.0000
8,HEO,1,1.0000
9,GEO,1,1.0000


,first_ground_success
0,2015-12-22 00:00:00


,LaunchSite,launches,mean_payload_kg
0,CCAFS SLC 40,55,5548.21
1,KSC LC 39A,22,7606.45
2,VAFB SLC 4E,13,5919.46


### Reconcile independent implementations

In [3]:
assert outputs['site_summary'].launches.sum() == len(df)
assert outputs['site_summary'].successes.sum() == df.Class.sum()
expected = group_summary(df, 'LaunchSite').sort_values('LaunchSite').reset_index(drop=True)
actual = outputs['site_summary'].sort_values('LaunchSite').reset_index(drop=True)
pd.testing.assert_series_equal(actual.launches, expected.launches, check_names=False)
print('SQL launch and success counts reconcile with Pandas.')

SQL launch and success counts reconcile with Pandas.


## Takeaways
SQL and Pandas summaries use the same launch-level population. Payload means inherit the course snapshot imputation; the table does not support customer-specific queries because customer is absent from this source schema.